[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/huggingface-nlp-certified/notebooks/day-02-tokenizers-deep-dive.ipynb#scrollTo=11223344)

---
# Day 2 · Tokenizers Deep Dive — BPE, WordPiece, and SentencePiece
**certified-journeys / huggingface-nlp-certified** · Day 2 · Core Concepts

> **Goal for today:** Understand how the three dominant subword tokenization algorithms work, compare their outputs on identical inputs, and confidently batch, pad, truncate, and decode token sequences.


## Step 1 · Install Dependencies

No new packages beyond Day 1 — `transformers` bundles the Rust-backed fast tokenizers via the `tokenizers` library (also by Hugging Face). The `sentencepiece` package is needed to load T5 and LLaMA tokenizers.


In [ ]:
%pip install -q transformers sentencepiece


## Step 2 · The Three Subword Algorithms

All modern NLP tokenizers split text into *subwords* rather than whole words or characters. This gives a bounded vocabulary that handles unseen words gracefully.

| Algorithm | Used by | Key idea |
|---|---|---|
| **WordPiece** | BERT, DistilBERT, ALBERT | Greedy merges that maximise language model likelihood |
| **BPE** (Byte-Pair Encoding) | GPT-2, RoBERTa, BART | Greedily merge most-frequent adjacent byte pairs |
| **SentencePiece** | T5, XLNet, LLaMA | Language-agnostic; treats raw bytes, adds `▁` for spaces |

In practice, the key differences you observe:
- WordPiece marks *continuation* subwords with `##` (e.g. `play ##ing`)
- BPE marks *word starts* with `Ġ` (Unicode space) and works at byte level
- SentencePiece marks word-start with `▁` (U+2581 LOWER ONE EIGHTH BLOCK)


In [ ]:
from transformers import AutoTokenizer

# Load BERT (WordPiece) and GPT-2 (BPE) tokenizers
bert_tok = AutoTokenizer.from_pretrained("bert-base-uncased")
gpt2_tok = AutoTokenizer.from_pretrained("gpt2")

sentence = "Tokenization is surprisingly interesting, especially for multilingual models!"

# Tokenize with each — just the tokens, not the full encoding
bert_tokens = bert_tok.tokenize(sentence)
gpt2_tokens = gpt2_tok.tokenize(sentence)

print(f"Original : {sentence}\n")
print(f"BERT  ({len(bert_tokens):>2} tokens): {bert_tokens}")
print()
print(f"GPT-2 ({len(gpt2_tokens):>2} tokens): {gpt2_tokens}")


### What just happened?

- **BERT** lowercases the input (uncased model) and uses `##` to mark subword continuations — `surprisingly` might split into `surprisingly` or `surprising ##ly` depending on vocabulary coverage.
- **GPT-2** preserves casing and marks word boundaries with `Ġ` (a visually distinct space character). `Ġtokenization` and `tokenization` are different tokens!
- **Token count differs**: the same sentence may tokenize into different numbers of tokens across models — critical for billing in API contexts.
- **Key insight:** Always use the tokenizer paired with your model. Mixing tokenizers and model weights produces garbage output.


## Step 3 · The Full Encoding — input_ids, attention_mask, token_type_ids

When you call a tokenizer as a function (not `.tokenize()`), you get a `BatchEncoding` with several tensors:

| Key | Meaning |
|---|---|
| `input_ids` | Integer ID of each token in the vocabulary |
| `attention_mask` | 1 for real tokens, 0 for padding |
| `token_type_ids` | 0 for sentence A, 1 for sentence B (BERT-style two-sentence tasks) |

Not all models use all three — GPT-2 only uses `input_ids`. BERT uses all three for tasks like NLI or QA.


In [ ]:
from transformers import AutoTokenizer

bert_tok = AutoTokenizer.from_pretrained("bert-base-uncased")

sentence = "The Hub has changed how engineers ship NLP models."

# Full encoding — returns a BatchEncoding (dict-like)
encoding = bert_tok(sentence)

print("Keys returned:", list(encoding.keys()))
print()
print(f"input_ids       ({len(encoding['input_ids'])} tokens):")
print(" ", encoding["input_ids"])
print()
print(f"attention_mask  ({len(encoding['attention_mask'])} values):")
print(" ", encoding["attention_mask"])
print()
print(f"token_type_ids  ({len(encoding['token_type_ids'])} values):")
print(" ", encoding["token_type_ids"])

# Show id → token mapping
print("\nID → Token mapping:")
tokens = bert_tok.convert_ids_to_tokens(encoding["input_ids"])
for token_id, token in zip(encoding["input_ids"], tokens):
    print(f"  {token_id:>6}  {token}")


### What just happened?

- **Special tokens**: BERT prepends `[CLS]` (ID 101) and appends `[SEP]` (ID 102) automatically — these are not in your original sentence.
- **`attention_mask` is all 1s** for a single sentence without padding. It becomes significant once you batch and pad.
- **`token_type_ids` is all 0s** for a single sentence; it becomes `[0…0, 1…1]` for two-sentence inputs.
- **Key insight:** The `[CLS]` token's hidden state is used as the sequence-level representation for classification tasks — it's not a throwaway token.


## Step 4 · Batching with Padding and Truncation

Real-world batches contain sentences of different lengths. You must pad shorter sequences and truncate longer ones so all tensors in a batch have the same shape.

- `padding=True` — pad to the longest sequence in the batch (`"longest"`) or to `max_length`
- `truncation=True` — truncate sequences exceeding `max_length`
- `max_length=128` — common default; BERT's absolute max is 512
- `return_tensors="pt"` — return PyTorch tensors; `"tf"` for TensorFlow, `"np"` for NumPy


In [ ]:
from transformers import AutoTokenizer

bert_tok = AutoTokenizer.from_pretrained("bert-base-uncased")

texts = [
    "Short sentence.",
    "A much longer sentence that has significantly more words and will need less padding.",
    "Medium length sentence here.",
]

# Batch encode with padding + truncation
batch = bert_tok(
    texts,
    padding=True,        # pad to longest in batch
    truncation=True,     # truncate if > max_length
    max_length=128,
    return_tensors="pt", # PyTorch tensors
)

print("Batch tensor shapes:")
for key, tensor in batch.items():
    print(f"  {key:<20}: {tuple(tensor.shape)}")

print("\nAttention mask (1=real token, 0=padding):")
for i, (text, mask) in enumerate(zip(texts, batch["attention_mask"])):
    real_count = mask.sum().item()
    pad_count  = len(mask) - real_count
    print(f"  [{i}] real={int(real_count):>3}, pad={int(pad_count):>3}  |  '{text[:45]}...' ")


### What just happened?

- All three tensors (`input_ids`, `attention_mask`, `token_type_ids`) have shape `[batch_size, max_seq_len]` — the model expects equal-length rows.
- **Padding token** is `[PAD]` (ID 0 for BERT). The `attention_mask` zeros out padding positions so the model ignores them.
- **Right-padding** is the default; BERT's bidirectional attention makes position of padding irrelevant as long as the mask is correct.
- **Key insight:** The `attention_mask` is load-bearing — forgetting it causes the model to attend to padding tokens and degrades accuracy, especially for short sentences in long-padded batches.


## Step 5 · Decoding — Token IDs Back to Strings

Decoding converts integer IDs back to human-readable text. Two methods:

- `tokenizer.decode(ids)` — decode a single sequence (list of ints)
- `tokenizer.batch_decode(batch_ids)` — decode a 2-D array all at once

`skip_special_tokens=True` removes `[CLS]`, `[SEP]`, `[PAD]` from the output — usually what you want when displaying results.


In [ ]:
from transformers import AutoTokenizer

bert_tok = AutoTokenizer.from_pretrained("bert-base-uncased")

# Encode, then decode round-trip
original = "Decoding converts token IDs back to human-readable strings."
ids = bert_tok.encode(original)  # returns list of ints (includes special tokens)

print("=== Single decode ===")
print(f"Original  : {original}")
print(f"IDs       : {ids}")
print(f"Decoded   : {bert_tok.decode(ids, skip_special_tokens=True)}")
print(f"With toks : {bert_tok.decode(ids, skip_special_tokens=False)}")

# Batch decode
sentences = [
    "Batch decoding is efficient.",
    "It handles multiple sequences at once.",
    "Padding tokens are stripped automatically.",
]
batch_ids = bert_tok(sentences, padding=True)["input_ids"]

print("\n=== Batch decode ===")
decoded = bert_tok.batch_decode(batch_ids, skip_special_tokens=True)
for orig, dec in zip(sentences, decoded):
    match = "✓" if orig.lower() == dec.strip().lower() else "≠"
    print(f"  {match} '{dec}'")


### What just happened?

- **`encode` vs `__call__`**: `encode()` returns a plain list of ints; calling the tokenizer returns a `BatchEncoding` with all keys. Use `encode` only for quick inspection.
- **Round-trip fidelity**: Uncased BERT lowercases input, so decoding does not recover the original case — this is a lossy transformation by design.
- **`skip_special_tokens=True`** removes `[CLS]` and `[SEP]` — critical for generation tasks where the model output must be clean text.
- **Key insight:** `batch_decode` is vectorised; prefer it over a Python loop over `decode()` when processing large outputs (e.g. generation results).


## Step 6 · Vocabulary Size and Special Tokens Across Models

Vocabulary size directly affects model size (the embedding matrix is `vocab_size × hidden_dim`). Different models make different trade-offs:

| Model | Algorithm | Vocab size | Notes |
|---|---|---|---|
| BERT base | WordPiece | 30,522 | English-focused |
| GPT-2 | BPE | 50,257 | Byte-level BPE |
| T5 base | SentencePiece | 32,100 | Multilingual-lite |
| mBERT | WordPiece | 119,547 | 104 languages |


In [ ]:
from transformers import AutoTokenizer

model_ids = [
    "bert-base-uncased",
    "gpt2",
    "t5-small",
]

print(f"{'Model':<30} {'Vocab':>8}  {'Special tokens'}")
print("-" * 70)

for model_id in model_ids:
    tok = AutoTokenizer.from_pretrained(model_id)
    special = tok.all_special_tokens  # list of special token strings
    print(f"{model_id:<30} {tok.vocab_size:>8,}  {special[:6]}")

# Deep dive: inspect the BERT special token IDs
bert_tok = AutoTokenizer.from_pretrained("bert-base-uncased")
print("\n=== BERT special token IDs ===")
print(f"  [CLS]  id={bert_tok.cls_token_id}  token='{bert_tok.cls_token}'")
print(f"  [SEP]  id={bert_tok.sep_token_id}  token='{bert_tok.sep_token}'")
print(f"  [PAD]  id={bert_tok.pad_token_id}  token='{bert_tok.pad_token}'")
print(f"  [UNK]  id={bert_tok.unk_token_id}  token='{bert_tok.unk_token}'")
print(f"  [MASK] id={bert_tok.mask_token_id} token='{bert_tok.mask_token}'")


### What just happened?

- GPT-2's vocabulary (50,257) is larger than BERT's (30,522) primarily because byte-level BPE guarantees it can represent any UTF-8 text without `[UNK]`.
- **T5** uses a relative vocabulary size (~32k) but covers more languages than BERT-base because SentencePiece selects subwords by frequency across a multi-lingual corpus.
- **Fast tokenizers** (the default when available) are backed by the Rust `tokenizers` library — 10–100x faster than the Python-fallback slow tokenizers, especially critical for batches of thousands of examples.
- **Key insight:** Vocabulary size is a hyperparameter baked into the pre-trained checkpoint — you cannot change it without retraining from scratch.


## Challenge

Tokenize the same multilingual sentence in three languages (English, French, German) using both `bert-base-multilingual-cased` and `gpt2` (English-only BPE). Compare the token counts, then decode the mBERT tokens and verify the round-trip.


In [ ]:
# Challenge: Cross-lingual tokenization comparison
# Your solution here

from transformers import AutoTokenizer

sentences = {
    "en": "The quick brown fox jumps over the lazy dog.",
    "fr": "Le renard brun rapide saute par-dessus le chien paresseux.",
    "de": "Der schnelle braune Fuchs springt über den faulen Hund.",
}

# Step 1: Load mBERT and GPT-2 tokenizers
# mbert = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")
# gpt2  = AutoTokenizer.from_pretrained("gpt2")

# Step 2: Tokenize each sentence with both tokenizers, print token counts
# for lang, text in sentences.items():
#     mbert_toks = ...
#     gpt2_toks  = ...
#     print(f"{lang}: mBERT={len(mbert_toks)}, GPT-2={len(gpt2_toks)}")

# Step 3: Decode the mBERT tokens for each language and verify round-trip
# (hint: mBERT is cased — does the round-trip preserve case?)


---
## Day 2 Key Concepts Recap

| Concept | What to remember |
|---|---|
| WordPiece | Marks continuations with `##`; used by BERT |
| BPE | Byte-level, marks word-starts with `Ġ`; used by GPT-2 |
| SentencePiece | Language-agnostic, marks word-starts with `▁`; used by T5 |
| `input_ids` | Integer IDs for each token, including special tokens |
| `attention_mask` | 1 = real token, 0 = padding — never skip it |
| `padding=True, truncation=True` | Always pass both when batching |
| Fast tokenizers | Rust-backed; default when available; 10–100x faster |
| Vocab size | Baked into checkpoint; cannot change without retraining |

> **Tip:** Use fast tokenizers (the default) — they are backed by Rust and are 10-100x faster than slow Python tokenizers, especially for large batches.

---
## What's next
**Day 3** → The Pipeline API — Classification, NER, Summarization, and More. You will benchmark inference time and learn how to use every major task in the pipeline zoo.

Mark Day 2 complete in your [tracker](../index.html).
